# Internações por mês e por estado, e previsão com LSTM

Cada ponto da série é o número de AIHs normais (`IDENT` = 1) cuja **data de internação** (`DT_INTER`) cai naquele mês, no **estado do hospital**. Os arquivos do SIH vêm organizados pelo mês de faturamento, que não é o mês da internação: uma AIH de janeiro pode ser lançada meses depois. Por isso o notebook lê os arquivos e reagrupa pela data de internação.

Arquivos usados: SIH-RD, janeiro de 2008 a setembro de 2026, as 27 UFs. A série que entra no modelo vai de **julho de 2008 a março de 2026**. Os seis primeiros meses compensam internações de 2008 lançadas ainda em 2007, e os seis últimos ficam de fora porque ainda vão receber lançamentos atrasados.

A LSTM prevê os **12 meses seguintes** de cada estado. Primeiro ela é avaliada nos últimos 12 meses já conhecidos (abril de 2025 a março de 2026). Depois é reestimada com toda a série confiável e projeta **abril de 2026 a março de 2027**.

A leitura baixa um arquivo por UF e mês. São cerca de 6 mil arquivos e a primeira execução leva horas. O que já foi contado fica em `dados/contagens/` e não é baixado de novo.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import nest_asyncio
import numpy as np
import pandas as pd
import torch
from torch import nn

nest_asyncio.apply()

from datasus import UFS, internacoes

PASTA = Path("dados")
CONTAGENS = PASTA / "contagens"
CONTAGENS.mkdir(parents=True, exist_ok=True)

ANO_INICIO = 2008
ANO_FIM = 2026
MES_FIM = 9
MESES_BORDA = 6
JANELA = 24
HORIZONTE = 12
SEMENTE = 42

torch.manual_seed(SEMENTE)
np.random.seed(SEMENTE)
plt.rcParams["figure.figsize"] = (11, 4)
print(f"{len(UFS)} UFs. Contagens em {CONTAGENS.resolve()}")

In [ ]:
def meses_do_ano(ano: int) -> list[int]:
    if ano < ANO_FIM:
        return list(range(1, 13))
    if ano == ANO_FIM:
        return list(range(1, MES_FIM + 1))
    return []


def baixar_ano(uf: str, ano: int) -> pd.DataFrame:
    meses = meses_do_ano(ano)
    try:
        return internacoes(
            uf=uf,
            ano=ano,
            meses=meses,
            grupo="RD",
            colunas=["DT_INTER", "IDENT"],
        )
    except Exception as erro_ano:
        print(f"  {uf} {ano}: o ano inteiro falhou ({erro_ano}). Tentando mês a mês.")
        partes = []
        for mes in meses:
            try:
                partes.append(
                    internacoes(
                        uf=uf,
                        ano=ano,
                        meses=mes,
                        grupo="RD",
                        colunas=["DT_INTER", "IDENT"],
                    )
                )
            except Exception as erro_mes:
                print(f"  {uf} {ano}-{mes:02d} falhou: {erro_mes}")
        if not partes:
            raise RuntimeError(f"Nenhum mês de {uf} {ano} foi lido.") from erro_ano
        return pd.concat(partes, ignore_index=True)


def contar_ano(uf: str, ano: int) -> pd.DataFrame:
    destino = CONTAGENS / f"{uf}_{ano}.parquet"
    if destino.exists():
        return pd.read_parquet(destino)

    bruto = baixar_ano(uf, ano)
    vazio = pd.DataFrame(columns=["uf", "mes", "internacoes"])
    if bruto.empty:
        vazio.to_parquet(destino, index=False)
        return vazio

    ident = bruto["IDENT"].astype(str).str.strip().str.lstrip("0")
    datas = bruto.loc[ident.eq("1"), "DT_INTER"].astype(str).str.slice(0, 6)
    datas = datas[datas.str.fullmatch(r"(?:19|20)\d{4}")]
    if datas.empty:
        vazio.to_parquet(destino, index=False)
        return vazio

    contagem = (
        datas.value_counts()
        .rename_axis("mes")
        .reset_index(name="internacoes")
    )
    contagem["uf"] = uf
    contagem = contagem[["uf", "mes", "internacoes"]]
    contagem.to_parquet(destino, index=False)
    return contagem

## Leitura

A célula abaixo percorre as 27 UFs e cada ano de 2008 a 2026. Pode interromper e rodar de novo: o arquivo `dados/contagens/UF_ANO.parquet` guarda a contagem daquele ano.

In [ ]:
for uf in UFS:
    for ano in range(ANO_INICIO, ANO_FIM + 1):
        destino = CONTAGENS / f"{uf}_{ano}.parquet"
        if destino.exists():
            continue
        print(f"{uf} {ano}")
        contar_ano(uf, ano)

partes = [pd.read_parquet(caminho) for caminho in sorted(CONTAGENS.glob("*.parquet"))]
contagens = pd.concat(partes, ignore_index=True)
contagens = contagens[contagens["mes"].between("200801", "202609")]
serie = (
    contagens.groupby(["uf", "mes"], as_index=False)["internacoes"]
    .sum()
)
serie["mes"] = pd.to_datetime(serie["mes"], format="%Y%m")
serie.to_parquet(PASTA / "internacoes_mes_uf.parquet", index=False)
print(f"{len(serie):,} linhas UF-mês com internação registrada")
serie.head()

In [ ]:
def painel_mensal(serie_longa: pd.DataFrame, inicio: str, fim: str) -> pd.DataFrame:
    largo = serie_longa.pivot(index="mes", columns="uf", values="internacoes")
    largo = largo.reindex(columns=list(UFS))
    indice = pd.date_range(inicio, fim, freq="MS")
    return largo.reindex(indice).fillna(0).astype(int)


observado = painel_mensal(serie, "2008-01-01", "2026-09-01")
confiavel = observado.loc["2008-07-01":"2026-03-01"]
print(f"Série confiável: {confiavel.index.min():%Y-%m} a {confiavel.index.max():%Y-%m}")
print(f"{confiavel.shape[0]} meses × {confiavel.shape[1]} estados")
confiavel.tail()

In [ ]:
fig, eixos = plt.subplots(9, 3, figsize=(12, 16), sharex=True)
for eixo, uf in zip(eixos.ravel(), UFS):
    eixo.plot(confiavel.index, confiavel[uf], color="#1f4e79", linewidth=0.8)
    eixo.set_title(uf, loc="left", fontsize=9)
    eixo.tick_params(labelsize=7)
fig.suptitle("Internações por mês, estado do hospital", y=1.01)
fig.tight_layout()
fig.savefig(PASTA / "serie_por_estado.png", dpi=120, bbox_inches="tight")
plt.show()

## LSTM

Um único modelo atende os 27 estados. Cada exemplo é uma janela de 24 meses de um estado. As entradas de cada mês são a quantidade padronizada daquele estado, o seno e o cosseno do mês do ano. A saída são os 12 meses seguintes.

A avaliação usa abril de 2025 a março de 2026, que o treino não vê. A previsão publicada é outra estimação, feita com a série confiável inteira, para abril de 2026 a março de 2027.

In [ ]:
class EscalaEstado:
    def __init__(self, treino: pd.DataFrame):
        self.minimo = treino.min()
        self.maximo = treino.max().clip(lower=self.minimo + 1)

    def aplicar(self, valores: pd.DataFrame) -> pd.DataFrame:
        return (valores - self.minimo) / (self.maximo - self.minimo)

    def desfazer(self, uf: str, valores: np.ndarray) -> np.ndarray:
        saida = valores * (self.maximo[uf] - self.minimo[uf]) + self.minimo[uf]
        return np.clip(saida, 0, None)


def calendario(indice: pd.DatetimeIndex) -> np.ndarray:
    angulo = 2 * np.pi * indice.month.to_numpy() / 12
    return np.column_stack([np.sin(angulo), np.cos(angulo)])


def montar_janelas(escalado: pd.DataFrame, t_inicial: int, t_final: int):
    cal = calendario(escalado.index)
    entradas, saidas, estados = [], [], []
    for uf in escalado.columns:
        serie_uf = escalado[uf].to_numpy(dtype=np.float32)
        for t in range(t_inicial, t_final + 1):
            quantidade = serie_uf[t - JANELA : t][:, None]
            entradas.append(np.hstack([quantidade, cal[t - JANELA : t]]))
            saidas.append(serie_uf[t : t + HORIZONTE])
            estados.append(uf)
    x = torch.tensor(np.stack(entradas), dtype=torch.float32)
    y = torch.tensor(np.stack(saidas), dtype=torch.float32)
    return x, y, estados


class LSTMInternacoes(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=3, hidden_size=64, batch_first=True)
        self.saida = nn.Linear(64, HORIZONTE)

    def forward(self, x):
        sequencia, _ = self.lstm(x)
        return self.saida(sequencia[:, -1, :])


def treinar(x, y, epocas=40):
    modelo = LSTMInternacoes()
    otimizador = torch.optim.Adam(modelo.parameters(), lr=1e-3)
    perda = nn.MSELoss()
    modelo.train()
    for epoca in range(1, epocas + 1):
        otimizador.zero_grad()
        erro = perda(modelo(x), y)
        erro.backward()
        otimizador.step()
        if epoca == 1 or epoca % 10 == 0:
            print(f"época {epoca:02d}  erro {erro.item():.5f}")
    modelo.eval()
    return modelo

In [ ]:
n_meses = len(confiavel)
t_teste = n_meses - HORIZONTE
escala_avaliacao = EscalaEstado(confiavel.iloc[:t_teste])
escalado_avaliacao = escala_avaliacao.aplicar(confiavel)

x_treino, y_treino, _ = montar_janelas(
    escalado_avaliacao, JANELA, t_teste - HORIZONTE
)
x_teste, y_teste, ufs_teste = montar_janelas(
    escalado_avaliacao, t_teste, t_teste
)
print(f"Treino: {len(x_treino):,} janelas. Teste: {len(x_teste)} estados.")

modelo_teste = treinar(x_treino, y_treino)
with torch.no_grad():
    previsto_teste = modelo_teste(x_teste).numpy()

meses_teste = confiavel.index[-HORIZONTE:]
linhas = []
for i, uf in enumerate(ufs_teste):
    real = confiavel[uf].iloc[-HORIZONTE:].to_numpy()
    previsto = np.rint(escala_avaliacao.desfazer(uf, previsto_teste[i])).astype(int)
    linhas.append(
        {
            "uf": uf,
            "mae": np.mean(np.abs(previsto - real)),
            "mape_%": np.mean(np.abs(previsto - real) / np.maximum(real, 1)) * 100,
        }
    )
avaliacao = pd.DataFrame(linhas).set_index("uf")
print(
    f"MAE médio {avaliacao['mae'].mean():,.0f} internações/mês. "
    f"MAPE médio {avaliacao['mape_%'].mean():.1f}%"
)
avaliacao.round(1)

In [ ]:
escala_final = EscalaEstado(confiavel)
escalado_final = escala_final.aplicar(confiavel)
x_final, y_final, _ = montar_janelas(
    escalado_final, JANELA, n_meses - HORIZONTE
)
modelo_final = treinar(x_final, y_final)

cal = calendario(confiavel.index)
ultimas = []
for uf in confiavel.columns:
    quantidade = escalado_final[uf].to_numpy(dtype=np.float32)[-JANELA:, None]
    ultimas.append(np.hstack([quantidade, cal[-JANELA:]]))
x_futuro = torch.tensor(np.stack(ultimas), dtype=torch.float32)

with torch.no_grad():
    bruto = modelo_final(x_futuro).numpy()

meses_futuros = pd.date_range(
    confiavel.index.max() + pd.DateOffset(months=1), periods=HORIZONTE, freq="MS"
)
previsao = pd.DataFrame(index=meses_futuros, columns=list(UFS), dtype=int)
for i, uf in enumerate(confiavel.columns):
    previsao[uf] = np.rint(escala_final.desfazer(uf, bruto[i])).astype(int)

previsao.to_parquet(PASTA / "previsao_12m.parquet")
print(f"Previsão de {previsao.index.min():%Y-%m} a {previsao.index.max():%Y-%m}")
previsao

In [ ]:
fig, eixos = plt.subplots(2, 2, figsize=(12, 6), sharex=True)
for eixo, uf in zip(eixos.ravel(), ["SP", "MG", "BA", "AM"]):
    historia = confiavel[uf].iloc[-36:]
    eixo.plot(historia.index, historia, color="#1f4e79", label="observado")
    eixo.plot(
        previsao.index,
        previsao[uf],
        color="#c45911",
        marker="o",
        label="previsto",
    )
    eixo.set_title(uf, loc="left")
    eixo.legend(fontsize=8)
fig.suptitle("Últimos 36 meses e previsão dos 12 seguintes")
fig.tight_layout()
fig.savefig(PASTA / "previsao_exemplos.png", dpi=120, bbox_inches="tight")
plt.show()